# 第 2 课：三维轨迹运动学特征

目标：将 `[时间, x/y/z]` 位置序列变成可解释的九维运动学特征。

In [ ]:
import sys
from pathlib import Path

# 同时兼容：从仓库根目录启动 Jupyter，或从 notebooks/ 目录启动。
search_starts = [Path.cwd(), *Path.cwd().parents]
repo_root = next((path for path in search_starts if (path / "pyproject.toml").exists()), None)
if repo_root is None:
    raise RuntimeError("没有找到 pyproject.toml；请从仓库目录启动 Jupyter。")

src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

print(f"仓库根目录: {repo_root}")
print(f"Python: {sys.version.split()[0]}")

## 2.1 准备一个历史窗口

核心源码：[features.py](../src/memcast_uav/features.py)  
文字讲解：[02_features.md](../tutorial/02_features.md)

In [ ]:
from memcast_uav.data import make_synthetic_flight, make_train_test_windows
from memcast_uav.features import (
    FEATURE_NAMES,
    extract_motion_features,
    translation_invariant_path,
)

flight = make_synthetic_flight(n_points=360)
_, test = make_train_test_windows(flight, split_index=252)
window = test[0]
print("历史窗口形状:", window.history.shape)

## 2.2 从位置差分得到速度、加速度和转弯率

In [ ]:
import numpy as np

velocity = np.diff(window.history, axis=0) / window.dt
acceleration = np.diff(velocity, axis=0) / window.dt
heading = np.unwrap(np.arctan2(velocity[:, 1], velocity[:, 0]))
turn_rate = np.diff(heading) / window.dt

print("速度形状:", velocity.shape, "单位 m/s")
print("加速度形状:", acceleration.shape, "单位 m/s²")
print("转弯率形状:", turn_rate.shape, "单位 rad/s")

## 2.3 调用正式特征提取函数

In [ ]:
features = extract_motion_features(window.history, window.dt)
units = ["m/s", "m/s", "m/s", "m/s²", "m/s²", "m/s", "rad/s", "m", "m"]

for name, value, unit in zip(FEATURE_NAMES, features, units, strict=True):
    print(f"{name:>24}: {value:9.4f} {unit}")
print("特征形状:", features.shape)

## 2.4 验证相对轨迹不受整体平移影响

In [ ]:
shifted = window.history + np.array([100.0, -20.0, 7.0])
same = np.allclose(
    translation_invariant_path(window.history),
    translation_invariant_path(shifted),
)
print("整体平移后相对轨迹相同:", same)
assert same

## 2.5 分块练习：观察量纲放大的影响

In [ ]:
scaled = features.copy()
scaled[-1] *= 1000
print("原始 altitude_change:", features[-1])
print("人为放大后:", scaled[-1])
print("TODO：思考为什么真实数据必须只用训练集统计量进行标准化。")

## 2.6 本课验收

In [ ]:
import subprocess

completed = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_features.py", "-q"],
    cwd=repo_root,
    check=True,
    text=True,
    capture_output=True,
)
print(completed.stdout)

[← 第 1 课](01_windowing.ipynb) · [教程目录](README.md) · [下一课：组合检索 →](03_retrieval.ipynb)